# មេរៀនទី 11 - ពិធីការភ្នាក់ងារទៅភ្នាក់ងារ (A2A)


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## ការបង្កើតភ្នាក់ងារធ្វើដំណើរដែលមានជំនាញពិសេស


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ការសហការរវាងភ្នាក់ងារច្រើនតាមរយៈលំហូរខ្សែការងារ

យើងភ្ជាប់ភ្នាក់ងារត្រីរូបទៅក្នុងលំហូរខ្សែការងារតាមលំដាប់ដែលសម្រមូលទៅនឹងការផ្ញើសារ A2A:

1. **CurrencyExchangeAgent** ទទួលសំណើរពីអ្នកប្រើ ហើយផលិតនូវការណែនាំអំពីរូបិយវត្ថុ។
2. **ActivityPlannerAgent** ទទួលសេចក្ដីបរិយាយដែលបានបន្ថែម ហើយបន្ថែមការប្រាប់បន្ថែមអំពីសកម្មភាព។
3. **TravelManagerAgent** សម្រួលទាំងពីរបញ្ចូលទៅជាការណែនាំធ្វើដំណើរចុងក្រោយ។


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Understanding A2A in Production

In a production environment the A2A protocol unlocks powerful cross-service scenarios:

| Capability | Description |
|---|---|
| **Cross-framework interop** | An agent built with one framework can delegate tasks to an agent built with any other A2A-compliant framework, enabling true cross-organization interoperability. |
| **Service boundaries** | Agents can live in separate microservices, cloud regions, or even different organisations while still collaborating seamlessly. |
| **Dynamic discovery** | An orchestrator can query an Agent Card registry at runtime to find the best-suited specialist for a given sub-task. |
| **Streaming & push notifications** | A2A supports Server-Sent Events (SSE) for real-time progress updates and push notifications for long-running tasks. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## សេចក្ដីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀន៖

1. **តើប្រព័ន្ធ A2A ជាអ្វី** — ស្តង់ដារបើកសម្រាប់ការស្វែងរកភ្នាក់ងារ-ទៅ-ភ្នាក់ងារ, ការផ្ញើសារនិងការគ្រប់គ្រងភារកិច្ច។
   និងការគ្រប់គ្រងភារកិច្ច។
2. **របៀបបង្កើតភ្នាក់ងារជាក់លាក់** — ភ្នាក់ងារបម្លែងរូបិយវត្ថុ, ភ្នាក់ងាររៀបចំសកម្មភាព,
   និងអ្នករៀបចំដំណើរកំសាន្ត Travel Manager។
3. **របៀបភ្ជាប់ភ្នាក់ងារចូលទៅក្នុងដំណើរការ** — ប្រើ `WorkflowBuilder` សម្រាប់រៀបចំការផ្ញើសារដោយលំដាប់ជាជំហានរវាងភ្នាក់ងារ។
   ការផ្ញើសារដោយលំដាប់ជាជំហានរវាងភ្នាក់ងារ។
4. **របៀបដែល A2A ប្រតិបត្តិការ នាពេលផលិតកម្ម** — អនុញ្ញាតឱ្យមានការសហការកាត់ប្លង់ និងសេវាកម្ម ឆ្លងកាត់ការស្វែងរកឆាប់រហ័ស និងបន្ទាន់សម័យព័ត៌មានជាដំណើរការ។
   ជាមួយនឹងការស្វែងរកឆាប់រហ័ស និងបន្ទាន់សម័យការផ្លាស់ប្តូរ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
